# Evaluation Notebook
Evaluate a trained model against a testing dataset.

In [ ]:
import sys, os
project_dir = os.path.abspath('..')
if project_dir not in sys.path:
    sys.path.insert(0, project_dir)
print('Project Root:', project_dir)

In [ ]:
import torch
from src.age_gender_model.inference import AgeGenderPredictor
from src.training.dataset import AgeGenderDataModule
from tqdm import tqdm
from sklearn.metrics import accuracy_score, mean_absolute_error

model_path = "../model_store/ag_classifier_main_mobilenet_v3_large_aug_epoch24_loss1.5.pth"
predictor = AgeGenderPredictor(checkpoint_path=model_path)

In [ ]:
# Configure datamodule for testing
config = {'ds_path': '/content/data/UTKFace/UTKFace', 'batch_size': 32}
datamodule = AgeGenderDataModule(config, mode='test')
datamodule.setup('test')
test_loader = datamodule.test_dataloader()

In [ ]:
all_preds_gender = []
all_true_gender = []
all_preds_age = []
all_true_age = []

for batch in tqdm(test_loader):
    images, true_ages, true_genders, _, _ = batch
    images = images.to(predictor.device)
    
    with torch.no_grad():
        gender_logits, age_preds = predictor.model(images)
        gender_preds = torch.argmax(gender_logits, dim=1)
        
    all_preds_gender.extend(gender_preds.cpu().numpy())
    all_true_gender.extend(true_genders.numpy())
    all_preds_age.extend(age_preds.cpu().numpy())
    all_true_age.extend(true_ages.numpy())

print("Gender Accuracy:", accuracy_score(all_true_gender, all_preds_gender))
print("Age MAE:", mean_absolute_error(all_true_age, all_preds_age))